# S03 / S20 재실행 — Retrieval + LLM Judge

S03 (grade ≥ 2 문서 0개) 와 S20 에 대해 전체 파이프라인을 재실행한다.

1. BM25 / Dense / Hybrid retrieval (Top-40)
2. LLM relevance judge (0~3 grade)
3. `goldset_final.json`의 S03 / S20 항목 교체

완료 후 `goldset_final.json`을 덮어씌운다.  
`grade_j1` = 새 judge 결과, `grade_j2/j3` = `None`, `final_grade` = `grade_j1`, `confidence` = `"SINGLE"`

In [5]:
import os
os.chdir("/home/ubuntu/work/somin/backend")
os.getcwd()

'/home/ubuntu/work/somin/backend'

In [6]:
import asyncio
import json
import uuid
import time
import random
import requests
import nest_asyncio
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from app.modules.RAG.retriever import (
    full_bm25_with_onboarding,
    full_dense_with_onboarding,
    full_hybrid_with_onboarding,
)
from app.modules.RAG.anchor_book_pipeline import run_anchor_pipeline
from app.core.config import settings
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

nest_asyncio.apply()

/home/ubuntu/anaconda3/envs/libraian/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/anaconda3/envs/libraian/lib/python3.11/site-packages/elasticsearch/_sync/client/__init__.py:403: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [7]:
# ── 설정 ────────────────────────────────────────────────────────────────────
TARGET_SCENARIOS = ["S01", "S02", "S03", "S08", "S15", "S17"]   # 재실행 대상

CLOVA_API_KEY = settings.CLOVA_API_KEY

REPO_ROOT     = Path("/home/ubuntu/work/somin")
DATASET_DIR   = REPO_ROOT / "evaluation" / "dataset"
SCENARIO_PATH = DATASET_DIR / "scenario_data_after_retrieval.json"

# 재실행 전용 중간 결과 (resume 지원)
OUTPUT_JSONL  = DATASET_DIR / "goldset_s01_s02_s03_s08_s15_s17_rerun.jsonl"
OUTPUT_JSON   = DATASET_DIR / "goldset_s01_s02_s03_s08_s15_s17_rerun.json"

# 최종 덮어쓸 파일
FINAL_JSON    = DATASET_DIR / "goldset_final.json"

URL         = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"
CONCURRENCY = 8
MAX_RETRIES = 5
LIMIT       = None   # 테스트 시 10 으로 변경

with open(SCENARIO_PATH, encoding="utf-8") as f:
    all_scenarios = json.load(f)

# 대상 시나리오만 필터링
scenarios = [s for s in all_scenarios if s["scenario_id"] in TARGET_SCENARIOS]
print(f"전체 시나리오: {len(all_scenarios)}  /  재실행 대상: {[s['scenario_id'] for s in scenarios]}")

전체 시나리오: 21  /  재실행 대상: ['S01', 'S02', 'S03', 'S08', 'S15', 'S17']


In [8]:
# ── Retrieval 유틸 ────────────────────────────────────────────────────────
def extract_rag_query(scenario: dict) -> dict:
    rag_query = scenario["turns"][-1]["rag_query"].copy()
    rag_query["query_id"] = scenario["scenario_id"]
    return rag_query


def simplify_item(item, source_name):
    return {
        "isbn":         item.get("isbn"),
        "title":        item.get("title"),
        "author":       item.get("author"),
        "publisher":    item.get("publisher"),
        "publish_date": item.get("publish_date"),
        "page":         item.get("page"),
        "large_cate":   item.get("large_cate"),
        "mid_cate":     item.get("mid_cate"),
        "small_cate":   item.get("small_cate"),
        "book_intro":   item.get("book_intro"),
        "book_index":   item.get("book_index"),
        "review_count": item.get("review_count"),
        "review_text":  item.get("review_text"),
    }


def merge_with_sources(result_groups, key="isbn"):
    merged = {}
    for source_name, results in result_groups.items():
        for item in results:
            item_key = item.get(key)
            if item_key is None:
                continue
            if item_key not in merged:
                merged[item_key] = simplify_item(item, source_name)
                merged[item_key]["retrieval_info"] = []
            merged[item_key]["retrieval_info"].append({
                "source": source_name,
                "rank":   item.get("rank"),
                "score":  item.get("score"),
            })
    return list(merged.values())

In [9]:
# ── LLM Judge ────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """
당신은 매우 보수적으로 책 추천 적합성을 판단하는 평가자입니다.

입력으로는 다음 두 정보가 주어집니다.

1. request: 사용자의 검색 요청 전체
- keyword_query
- semantic_query
- filters
- constraints
- score_boost
- session_signals
- onboarding_signals

2. book: 추천 후보 도서 정보

[판정 기준]
- 현재 사용자 요청의 핵심 의도와 조건을 우선적으로 판단하세요.
- session_signals를 최우선 기준으로 판단하세요.
- onboarding_signals는 session_signals와 충돌하지 않는 경우에만 보조적으로 반영하세요.
- session_signals와 onboarding_signals가 충돌하면 session_signals를 우선하세요.
- 사용자 요청의 모든 표현이 책 정보에 직접 등장할 필요는 없습니다.
- 제목, 소개, 목차, 카테고리, 리뷰 등을 종합하여 의미적 적합성을 판단하세요.
- 책이 제공하는 독서 경험, 분위기, 주제, 대상 독자가 사용자 요청과 충분히 일치하면 높은 점수를 부여하세요.
- 일부 키워드만 우연히 일치하거나, 장르/주제/대상/분위기가 실제로 맞지 않으면 낮은 점수를 부여하세요.
- filters, constraints는 명시 조건으로 보고 우선 고려하세요.
- soft 조건은 약간 벗어나도 핵심 요청 적합성이 높으면 높은 점수를 부여할 수 있습니다.
- 정보가 부족하여 핵심 조건 충족 여부를 판단할 수 없으면 낮은 점수를 부여하세요.
- 요청과 직접 관련 없는 책이 단순히 키워드만 포함한 경우에는 낮은 점수를 부여하세요.
- retrieval rank, score, source 정보는 판단에 사용하지 마세요.
- 반드시 JSON만 출력하세요. 설명 문장, markdown, 코드블록은 출력하지 마세요.

[Grade 기준]
3 = 매우 적합. 현재 요청의 핵심 의도와 조건에 잘 부합함.
2 = 적합. 핵심 의도는 맞지만 일부 조건이 약함.
1 = 부분 관련. 관련성은 있으나 정답 도서로 보기에는 약함.
0 = 부적합. 현재 요청의 핵심 의도와 맞지 않음.

출력 형식:
{\n  \"relevance_grade\": 0 또는 1 또는 2 또는 3,\n  \"reason\": \"판정 근거를 간략히 설명하는 텍스트\"\n}
"""

EXCLUDE_KEYS = {
    "retrieval_info",
    "rank", "score", "source",
    "retrieval_source", "retrieval_rank", "retrieval_score",
    "bm25_score", "dense_score", "hybrid_score",
    "main_score", "onboarding_score",
    "disliked_penalty", "disliked_similarity",
    "relevance_grade", "binary_label", "label_status",
    "llm_raw_output", "llm_error", "query_id",
    "final_grade", "confidence", "grade_j1", "grade_j2", "grade_j3",
}


def make_headers():
    return {
        "Authorization": f"Bearer {CLOVA_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
    }


def make_book_for_judge(book):
    return {k: v for k, v in book.items()
            if k not in EXCLUDE_KEYS and not str(k).startswith("retrieval_")}


def parse_hcx_response(text):
    if "event:" in text:
        last_data = None
        for block in text.split("\n\n"):
            lines = block.strip().splitlines()
            event_type = data_text = None
            for line in lines:
                if line.startswith("event:"):
                    event_type = line.replace("event:", "").strip()
                elif line.startswith("data:"):
                    data_text = line.replace("data:", "").strip()
            if event_type == "result" and data_text:
                last_data = data_text
        if last_data is not None:
            data = json.loads(last_data)
            return data["message"]["content"]

    try:
        data = json.loads(text)
        if "result" in data:
            return data["result"]["message"]["content"]
        if "message" in data:
            return data["message"]["content"]
    except Exception:
        pass

    raise ValueError(f"응답 파싱 실패:\n{text[:1000]}")


def extract_grade(llm_text):
    try:
        obj = json.loads(llm_text)
        grade = obj.get("relevance_grade", obj.get("grade", obj.get("label")))
        if grade is None:
            return None, "parse_failed", "missing relevance_grade"
        grade = int(grade)
        if grade not in [0, 1, 2, 3]:
            return None, "parse_failed", f"invalid relevance_grade: {grade}"
        return grade, "success", None
    except Exception as e:
        return None, "parse_failed", str(e)


def grade_to_binary(grade):
    if grade is None:
        return None
    return 1 if grade >= 2 else 0


def make_user_prompt(request_data, book):
    payload = {
        "request": request_data,
        "book": make_book_for_judge(book),
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


def blocking_label_one_book(request_data, book, max_retries=MAX_RETRIES):
    query_id    = request_data.get("query_id", "unknown")
    user_prompt = make_user_prompt(request_data, book)

    body = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        "topP": 0.1, "topK": 0, "max_tokens": 100,
        "temperature": 0.0, "repetitionPenalty": 1.0,
        "includeAiFilters": False, "seed": 42,
    }

    last_error = None
    for attempt in range(max_retries):
        try:
            res = requests.post(URL, headers=make_headers(), json=body, timeout=60)
            if res.status_code == 200:
                llm_text = parse_hcx_response(res.text)
                grade, label_status, parse_error = extract_grade(llm_text)
                return {
                    **book,
                    "query_id":        query_id,
                    "relevance_grade": grade,
                    "binary_label":    grade_to_binary(grade),
                    "label_status":    label_status,
                    "llm_raw_output":  llm_text,
                    "llm_error":       parse_error,
                }
            last_error = f"{res.status_code} / {res.text[:500]}"
            if res.status_code in [429, 500, 502, 503, 504]:
                raise Exception(last_error)
            return {
                **book,
                "query_id":        query_id,
                "relevance_grade": None,
                "binary_label":    None,
                "label_status":    "api_failed",
                "llm_raw_output":  None,
                "llm_error":       last_error,
            }
        except Exception as e:
            last_error = str(e)
            if attempt == max_retries - 1:
                return {
                    **book,
                    "query_id":        query_id,
                    "relevance_grade": None,
                    "binary_label":    None,
                    "label_status":    "api_failed",
                    "llm_raw_output":  None,
                    "llm_error":       last_error,
                }
            wait = min(30, (2 ** attempt) + random.uniform(0, 1))
            time.sleep(wait)


async def label_one_book_async(request_data, book, semaphore, write_lock, jsonl_file, pbar):
    async with semaphore:
        try:
            result = await asyncio.to_thread(blocking_label_one_book, request_data, book)
        except Exception as e:
            query_id = request_data.get("query_id", "unknown")
            result = {
                **book,
                "query_id":        query_id,
                "relevance_grade": None,
                "binary_label":    None,
                "label_status":    "api_failed",
                "llm_raw_output":  None,
                "llm_error":       str(e),
            }

    async with write_lock:
        jsonl_file.write(json.dumps(result, ensure_ascii=False) + "\n")
        jsonl_file.flush()

    pbar.update(1)
    return result

## Phase 1: Retrieval (S03, S20)

In [8]:
all_candidates: list = []

for scenario in tqdm(scenarios, desc="retrieval"):
    sample_result = extract_rag_query(scenario)

    # anchor_based 시나리오(S03)는 anchor 파이프라인으로 query 보강
    if sample_result.get("anchor"):
        sample_result = run_anchor_pipeline(sample_result)
    
    query_id = sample_result["query_id"]

    kw = sample_result.get("keyword_query") or []
    sq = sample_result.get("semantic_query") or ""
    
    print(f"\n[{query_id}]")
    print(f"  keyword_query : {kw}")
    print(f"  semantic_query: {sq}")

    re_bm25   = full_bm25_with_onboarding(sample_result, size=40)
    re_dense  = full_dense_with_onboarding(sample_result, size=40)
    re_hybrid = full_hybrid_with_onboarding(sample_result, size=40)

    full_books = merge_with_sources({
        "bm25":   re_bm25,
        "dense":  re_dense,
        "hybrid": re_hybrid,
    })

    for book in full_books:
        book["query_id"] = query_id
        all_candidates.append(book)

print(f"\n전체 후보 도서: {len(all_candidates):,}")
for sid in TARGET_SCENARIOS:
    cnt = sum(1 for b in all_candidates if b["query_id"] == sid)
    print(f"  {sid}: {cnt}")

retrieval:   0%|          | 0/6 [00:00<?, ?it/s]


[S01]
  keyword_query : ['실존주의', '철학적 사유', '삶의 지혜', '에세이', '인생 철학', '현명한 삶']
  semantic_query: 실존주의적 주제를 중심으로 삶의 의미에 대한 철학적 성찰을 유도하며, 현명한 삶의 태도를 탐구하는 에세이 추천


retrieval:  17%|█▋        | 1/6 [00:28<02:20, 28.15s/it]


[S02]
  keyword_query : ['경영 리더십 원칙', '경제 이론 비판', '현실적 대안', '실무 적용', '경제 경영 서적']
  semantic_query: 경영 리더십 원칙을 실무에 적용할 수 있도록 경제 이론을 비판하고 현실적 대안을 제시하는 경제 경영 서적


retrieval:  33%|███▎      | 2/6 [00:48<01:33, 23.50s/it]

...


[S15]
  keyword_query : ['심리학', '행동경제학', '실용적 적용', '동기부여', '일상 활용']
  semantic_query: 일상 생활에서 즉시 활용할 수 있는 심리학과 행동경제학 이론을 바탕으로 무기력함을 극복하고 동기부여를 얻을 수 있는 실용적인 책 추천


retrieval:  83%|████████▎ | 5/6 [01:39<00:19, 19.53s/it]


[S17]
  keyword_query : ['청소년 소설', '사회적 메시지', '성장 이야기', '따뜻한 분위기', '감동적인']
  semantic_query: 따뜻한 감동적인 분위기의 청소년 소설 중에서 사회적 메시지와 성장을 다룬 작품을 추천해주세요.


retrieval: 100%|██████████| 6/6 [01:58<00:00, 19.82s/it]


전체 후보 도서: 477
  S01: 75
  S02: 83
  S03: 70
  S08: 86
  S15: 80
  S17: 83


## Phase 2: LLM 라벨링 (비동기)

> 테스트: Cell 3의 `LIMIT = 10` 으로 설정 후 실행  
> 전체: `LIMIT = None`

In [9]:
async def main():
    query_map = {}
    for scenario in scenarios:
        rag_query = scenario["turns"][-1]["rag_query"].copy()
        rag_query["query_id"] = scenario["scenario_id"]
        query_map[scenario["scenario_id"]] = rag_query

    # resume: 이미 처리된 (query_id, isbn) 스킵
    processed_keys: set = set()
    already_labeled: list = []

    if OUTPUT_JSONL.exists():
        with OUTPUT_JSONL.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    item = json.loads(line)
                    key  = (item.get("query_id"), item.get("isbn"))
                    if key not in processed_keys:
                        processed_keys.add(key)
                        already_labeled.append(item)
                except json.JSONDecodeError:
                    continue
        print(f"이미 처리된 도서 수: {len(processed_keys):,}")

    to_process = [
        book for book in all_candidates
        if (book.get("query_id"), book.get("isbn")) not in processed_keys
        and book.get("query_id") in query_map
    ]

    if LIMIT is not None:
        to_process = to_process[:LIMIT]

    print(f"전체 후보: {len(all_candidates):,}  |  처리 대상: {len(to_process):,}")

    if not to_process:
        print("처리할 항목이 없습니다.")
        return already_labeled

    semaphore  = asyncio.Semaphore(CONCURRENCY)
    write_lock = asyncio.Lock()
    pbar       = tqdm(total=len(to_process), desc="라벨링 중", unit="book")

    with OUTPUT_JSONL.open("a", encoding="utf-8") as jsonl_file:
        tasks = [
            label_one_book_async(
                query_map[book["query_id"]],
                book,
                semaphore, write_lock, jsonl_file, pbar,
            )
            for book in to_process
        ]
        new_results = await asyncio.gather(*tasks)

    pbar.close()

    all_results = already_labeled + list(new_results)
    order = {(e.get("query_id"), e.get("isbn")): i for i, e in enumerate(all_candidates)}
    all_results.sort(key=lambda x: order.get((x.get("query_id"), x.get("isbn")), 99999))

    with OUTPUT_JSON.open("w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    print(f"\nJSONL: {OUTPUT_JSONL}")
    print(f"JSON : {OUTPUT_JSON}")

    stats = defaultdict(lambda: defaultdict(int))
    for item in all_results:
        if item.get("label_status") == "success":
            stats[item.get("query_id")][item.get("relevance_grade")] += 1

    for sid in TARGET_SCENARIOS:
        g = dict(sorted(stats[sid].items()))
        total = sum(g.values())
        pos   = sum(v for k, v in g.items() if k is not None and k >= 2)
        print(f"  {sid}: {g}  (grade≥2: {pos}/{total})")

    return all_results


results = await main()

전체 후보: 477  |  처리 대상: 477


라벨링 중: 100%|██████████| 477/477 [09:58<00:00,  1.25s/book]


JSONL: /home/ubuntu/work/somin/evaluation/dataset/goldset_s01_s02_s03_s08_s15_s17_rerun.jsonl
JSON : /home/ubuntu/work/somin/evaluation/dataset/goldset_s01_s02_s03_s08_s15_s17_rerun.json
  S01: {0: 5, 1: 64, 2: 5, 3: 1}  (grade≥2: 6/75)
  S02: {0: 13, 1: 67, 2: 3}  (grade≥2: 3/83)
  S03: {0: 21, 1: 47, 2: 2}  (grade≥2: 2/70)
  S08: {0: 5, 1: 73, 2: 7, 3: 1}  (grade≥2: 8/86)
  S15: {0: 15, 1: 63, 2: 2}  (grade≥2: 2/80)
  S17: {0: 7, 1: 49, 2: 20, 3: 7}  (grade≥2: 27/83)


## api_failed 재시도

In [10]:
async def retry_api_failed():
    if not OUTPUT_JSON.exists():
        print(f"파일 없음: {OUTPUT_JSON}")
        return []

    with OUTPUT_JSON.open("r", encoding="utf-8") as f:
        all_results = json.load(f)

    query_map = {}
    for scenario in scenarios:
        rag_query = scenario["turns"][-1]["rag_query"].copy()
        rag_query["query_id"] = scenario["scenario_id"]
        query_map[scenario["scenario_id"]] = rag_query

    to_retry = [
        e for e in all_results
        if e.get("label_status") in ("api_failed", "parse_failed")
        or e.get("relevance_grade") is None
    ]
    print(f"전체: {len(all_results):,}개  |  재시도 대상: {len(to_retry):,}개")

    if not to_retry:
        print("재시도할 항목이 없습니다.")
        return all_results

    semaphore = asyncio.Semaphore(CONCURRENCY)
    pbar      = tqdm(total=len(to_retry), desc="재시도 중", unit="book")

    async def retry_one(entry):
        query_id = entry.get("query_id", "unknown")
        if query_id not in query_map:
            pbar.update(1)
            return entry
        async with semaphore:
            try:
                result = await asyncio.to_thread(
                    blocking_label_one_book, query_map[query_id], entry
                )
            except Exception as e:
                result = {**entry, "label_status": "api_failed", "llm_error": str(e)}
        pbar.update(1)
        return result

    new_results = await asyncio.gather(*[retry_one(e) for e in to_retry])
    pbar.close()

    key_to_new = {(r.get("query_id"), r.get("isbn")): r for r in new_results}
    updated = [
        key_to_new.get((e.get("query_id"), e.get("isbn")), e)
        for e in all_results
    ]

    with OUTPUT_JSON.open("w", encoding="utf-8") as f:
        json.dump(updated, f, ensure_ascii=False, indent=2)

    now_success  = sum(1 for r in new_results if r.get("label_status") == "success")
    still_failed = sum(1 for r in new_results if r.get("label_status") in ("api_failed", "parse_failed"))
    print(f"재시도 완료: 성공 {now_success}  /  여전히 실패 {still_failed}")
    print(f"JSON: {OUTPUT_JSON}")
    return updated


results = await retry_api_failed()

전체: 477개  |  재시도 대상: 0개
재시도할 항목이 없습니다.


In [11]:
# 통계
stats      = defaultdict(lambda: defaultdict(int))
fail_stats = defaultdict(int)

for item in results:
    qid    = item.get("query_id", "unknown")
    grade  = item.get("relevance_grade")
    status = item.get("label_status")

    if status == "success":
        stats[qid][grade] += 1
    else:
        fail_stats[qid] += 1

total_success = sum(sum(g.values()) for g in stats.values())
total_failed  = sum(fail_stats.values())

print(f"{'='*68}")
print(f"재평가 완료")
print(f"{'='*68}")
print(f"총 항목: {len(results):,}")
print(f"성공:    {total_success:,} ({total_success/len(results)*100:.1f}%)")
print(f"실패:    {total_failed:,} ({total_failed/len(results)*100:.1f}%)")
print()

print(f"[query_id별 relevance_grade 분포]")
print(f"{'query_id':>10}  {'grade 0':>7}  {'grade 1':>7}  {'grade 2':>7}  {'grade 3':>7}  {'failed':>7}  {'total':>7}")
print("-" * 68)

all_qids = sorted(set(list(stats.keys()) + list(fail_stats.keys())))
for qid in all_qids:
    g      = stats[qid]
    failed = fail_stats[qid]
    total  = sum(g.values()) + failed
    print(f"{qid:>10}  {g.get(0,0):>7}  {g.get(1,0):>7}  {g.get(2,0):>7}  {g.get(3,0):>7}  {failed:>7}  {total:>7}")

재평가 완료
총 항목: 477
성공:    477 (100.0%)
실패:    0 (0.0%)

[query_id별 relevance_grade 분포]
  query_id  grade 0  grade 1  grade 2  grade 3   failed    total
--------------------------------------------------------------------
       S01        5       64        5        1        0       75
       S02       13       67        3        0        0       83
       S03       21       47        2        0        0       70
       S08        5       73        7        1        0       86
       S15       15       63        2        0        0       80
       S17        7       49       20        7        0       83


## goldset_final.json 업데이트

S03 / S20 의 기존 항목을 제거하고 새 결과로 교체한다.

- `grade_j1` = 새 judge 결과 (`relevance_grade`)
- `grade_j2`, `grade_j3` = `None` (단일 judge 재실행)
- `final_grade` = `grade_j1`
- `confidence` = `"SINGLE"`
- `label_status == "api_failed"` 항목은 제외

In [ ]:
# 새 결과 로드
with OUTPUT_JSON.open("r", encoding="utf-8") as f:
    new_results = json.load(f)

# 성공 항목만 사용 (api_failed/parse_failed 제외)
new_success = [r for r in new_results if r.get("label_status") == "success"]
print(f"새 결과 전체: {len(new_results)}  /  성공: {len(new_success)}")
for sid in TARGET_SCENARIOS:
    items = [r for r in new_success if r.get("query_id") == sid]
    from collections import Counter
    grades = Counter(r.get("relevance_grade") for r in items)
    pos = sum(v for k, v in grades.items() if k is not None and k >= 2)
    print(f"  {sid}: {dict(sorted(grades.items()))}  (grade≥2: {pos}/{len(items)})")

# goldset_final.json 로드
with FINAL_JSON.open("r", encoding="utf-8") as f:
    final_data = json.load(f)

print(f"\ngoldset_final.json 기존 항목: {len(final_data)}")
for sid in TARGET_SCENARIOS:
    cnt = sum(1 for r in final_data if r.get("query_id") == sid)
    print(f"  기존 {sid}: {cnt}건")

# S03 / S20 기존 항목 제거
kept = [r for r in final_data if r.get("query_id") not in TARGET_SCENARIOS]
print(f"\n제거 후 잔존: {len(kept)}")

# 새 항목 변환 (goldset_final 필드 구조로 맞춤)
def to_final_record(r):
    grade = r.get("relevance_grade")
    record = {
        k: v for k, v in r.items()
        if k not in ("llm_raw_output",)   # llm_raw_output은 공간 절약을 위해 제외
    }
    record["final_grade"]  = grade
    record["confidence"]   = "SINGLE"
    record["grade_j1"]     = grade
    record["grade_j2"]     = None
    record["grade_j3"]     = None
    return record

new_records = [to_final_record(r) for r in new_success]

# 병합 및 query_id 순서로 정렬
merged = kept + new_records
sid_order = {f"S{i:02d}": i for i in range(1, 30)}
merged.sort(key=lambda x: (sid_order.get(x.get("query_id", ""), 99), x.get("isbn", "")))

with FINAL_JSON.open("w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

print(f"\ngoldset_final.json 저장 완료: {len(merged)}건")
for sid in TARGET_SCENARIOS:
    cnt = sum(1 for r in merged if r.get("query_id") == sid)
    grades = Counter(r.get("final_grade") for r in merged if r.get("query_id") == sid)
    pos = sum(v for k, v in grades.items() if k is not None and k >= 2)
    print(f"  {sid}: {cnt}건  grade_dist={dict(sorted(grades.items()))}  (grade≥2: {pos})")

## 결과 검증

In [ ]:
with FINAL_JSON.open("r", encoding="utf-8") as f:
    final = json.load(f)

df = pd.DataFrame(final)

print(f"총 항목: {len(df):,}  /  query_id 수: {df['query_id'].nunique()}")

pivot = df.groupby(["query_id", "final_grade"]).size().unstack(fill_value=0)
pivot.columns = [f"g{int(c)}" for c in pivot.columns]
pivot["total"] = pivot.sum(axis=1)
grade2_cols = [c for c in pivot.columns if c in ("g2", "g3")]
pivot["g2+3"] = pivot[grade2_cols].sum(axis=1)

print("\n[query_id별 grade 분포]")
print(pivot.to_string())

insufficient = pivot[pivot["g2+3"] < 3]
if insufficient.empty:
    print("\n✅ 모든 query_id가 grade≥2 최소 3개 기준 충족")
else:
    print(f"\n⚠️  기준 미달 query_id: {list(insufficient.index)}")